In [1]:
!pip install transformers datasets scikit-learn pandas huggingface_hub


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import os
os.environ["HF_TOKEN"] = "<your_HF_Token>"

In [4]:
import os
import time
import torch
import pandas as pd
import numpy as np
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    get_cosine_schedule_with_warmup,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from huggingface_hub import HfApi, create_repo

print("=" * 55)
print(" IntelliGuard SPINE Training on AMD MI300X")
print("=" * 55)
print(f"ROCm: {torch.version.hip}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

device = torch.device("cuda")
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# ── Load dataset ──────────────────────────────────────
print("\nLoading dataset...")
ds_deepset = load_dataset("deepset/prompt-injections")
df_deepset = pd.concat([
    ds_deepset['train'].to_pandas(),
    ds_deepset['test'].to_pandas()
], ignore_index=True)

ds_hlyn = load_dataset("hlyn/prompt-injection-judge-deberta-dataset")
df_hlyn = ds_hlyn['train'].to_pandas().sample(n=5000, random_state=42)

df = pd.concat([df_deepset, df_hlyn], ignore_index=True)
df = df[['text', 'label']].dropna()
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f"Total samples: {len(df)}")
print(f"Label dist: {dict(df['label'].value_counts())}")

train_df, val_df = train_test_split(
    df, test_size=0.2, random_state=SEED, stratify=df['label']
)
print(f"Train: {len(train_df)} | Val: {len(val_df)}")

# ── Dataset class ─────────────────────────────────────
class SpineDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=128):
        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = str(self.data['text'][idx])
        label = int(self.data['label'][idx])
        enc = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'label': torch.tensor(label, dtype=torch.long)
        }

# ── Load model ────────────────────────────────────────
print("\nLoading DistilBERT...")
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=2
).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

# ── Dataloaders ───────────────────────────────────────
train_dataset = SpineDataset(train_df, tokenizer)
val_dataset = SpineDataset(val_df, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

# ── Optimizer ─────────────────────────────────────────
EPOCHS = 5
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = EPOCHS * len(train_loader)
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=int(total_steps * 0.06),
    num_training_steps=total_steps
)

# ── Training functions ────────────────────────────────
def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss, all_preds, all_labels = 0, [], []
    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        outputs.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += outputs.loss.item()
        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
    return total_loss/len(loader), accuracy_score(all_labels, all_preds), f1_score(all_labels, all_preds)

def eval_epoch(model, loader, device):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item()
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
    return total_loss/len(loader), accuracy_score(all_labels, all_preds), f1_score(all_labels, all_preds)

# ── Training loop ─────────────────────────────────────
print("\n" + "="*55)
print(" SPINE TRAINING — AMD MI300X ROCm")
print("="*55)

best_f1 = 0
for epoch in range(EPOCHS):
    t0 = time.time()
    train_loss, train_acc, train_f1 = train_epoch(model, train_loader, optimizer, scheduler, device)
    val_loss, val_acc, val_f1 = eval_epoch(model, val_loader, device)
    elapsed = time.time() - t0

    print(f"\nEpoch {epoch+1}/{EPOCHS} ({elapsed:.0f}s)")
    print(f"  Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}")
    print(f"  Val   Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | F1: {val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        os.makedirs("models/spine_amd", exist_ok=True)
        model.save_pretrained("models/spine_amd")
        tokenizer.save_pretrained("models/spine_amd")
        print(f"  ** Best SPINE saved (F1: {best_f1:.4f}) **")

print(f"\n{'='*55}")
print(f" FINAL SPINE F1: {best_f1:.4f}")
print(f" Trained on AMD MI300X via ROCm {torch.version.hip}")
print(f" Dataset: {len(df)} samples")
print(f"{'='*55}")

# ── Upload to HuggingFace ─────────────────────────────
print("\nUploading SPINE to HuggingFace...")
HF_TOKEN = os.environ.get("HF_TOKEN", "")
HF_USERNAME = "sarthak20P"
REPO_ID = f"{HF_USERNAME}/IntelliGuard-SPINE"

if HF_TOKEN:
    api = HfApi(token=HF_TOKEN)
    create_repo(repo_id=REPO_ID, repo_type="model", token=HF_TOKEN, exist_ok=True)
    api.upload_folder(
        folder_path="models/spine_amd",
        repo_id=REPO_ID,
        repo_type="model",
        token=HF_TOKEN
    )
    print(f"SPINE uploaded to: https://huggingface.co/{REPO_ID}")
else:
    print("No HF_TOKEN set — skipping upload")

print("Done!")

 IntelliGuard SPINE Training on AMD MI300X
ROCm: 7.0.51831-a3e329ad8
GPU: 
VRAM: 205.8 GB

Loading dataset...


Generating train split:   0%|          | 0/399741 [00:00<?, ? examples/s]

Total samples: 5662
Label dist: {0: 2912, 1: 2750}
Train: 4529 | Val: 1133

Loading DistilBERT...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

Parameters: 66,955,010
Train batches: 71 | Val batches: 18

 SPINE TRAINING — AMD MI300X ROCm

Epoch 1/5 (18s)
  Train Loss: 0.5224 | Acc: 0.7377 | F1: 0.7311
  Val   Loss: 0.3470 | Acc: 0.8429 | F1: 0.8227
  ** Best SPINE saved (F1: 0.8227) **

Epoch 2/5 (8s)
  Train Loss: 0.2656 | Acc: 0.8861 | F1: 0.8807
  Val   Loss: 0.2469 | Acc: 0.8932 | F1: 0.8940
  ** Best SPINE saved (F1: 0.8940) **

Epoch 3/5 (8s)
  Train Loss: 0.1748 | Acc: 0.9346 | F1: 0.9322
  Val   Loss: 0.2221 | Acc: 0.9153 | F1: 0.9111
  ** Best SPINE saved (F1: 0.9111) **

Epoch 4/5 (8s)
  Train Loss: 0.1161 | Acc: 0.9627 | F1: 0.9614
  Val   Loss: 0.2321 | Acc: 0.9153 | F1: 0.9098

Epoch 5/5 (8s)
  Train Loss: 0.0895 | Acc: 0.9737 | F1: 0.9728
  Val   Loss: 0.2228 | Acc: 0.9170 | F1: 0.9130
  ** Best SPINE saved (F1: 0.9130) **

 FINAL SPINE F1: 0.9130
 Trained on AMD MI300X via ROCm 7.0.51831-a3e329ad8
 Dataset: 5662 samples

Uploading SPINE to HuggingFace...


Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  .../models/spine_amd/model.safetensors:   2%|1         | 5.17MB /  268MB            

SPINE uploaded to: https://huggingface.co/sarthak20P/IntelliGuard-SPINE
Done!
